# SQL Complete Practice — Part 03: Constraints and Keys
**Topics 32–45 | MySQL | Ankanhaldar**

This notebook explains SQL constraints and database keys with separate theory and SQL examples. Work through the sections in order.

> **Execution note:** These cells contain MySQL SQL, not Python. Run them in a MySQL client such as VS Code with SQLTools, MySQL Workbench, or a SQL notebook/kernel. A standard Python kernel will not execute SQL directly.

> **Practice database:** The examples use the database `ankan_sql_practice`. Create it if needed, then select it before running the examples.

In [ ]:
CREATE DATABASE IF NOT EXISTS ankan_sql_practice;
USE ankan_sql_practice;

## Topic 32: What Are SQL Constraints?

A **constraint** is a rule applied to a table or column to control the data that can be stored. Constraints help maintain data accuracy, validity, and consistency.

Common MySQL constraints:
- `NOT NULL` — prevents a column from storing `NULL`.
- `UNIQUE` — prevents duplicate values in a column or column combination.
- `PRIMARY KEY` — uniquely identifies every row and does not allow `NULL`.
- `FOREIGN KEY` — enforces a valid relationship with a referenced table.
- `CHECK` — validates a condition.
- `DEFAULT` — supplies a value when one is not provided.
- `AUTO_INCREMENT` — generates sequential numeric values for a column.

Constraints are declared when creating a table or, where supported, added later using `ALTER TABLE`.

### SQL Example — Topic 32

In [ ]:
CREATE TABLE constraint_demo_overview (
    id INT PRIMARY KEY,
    name VARCHAR(60) NOT NULL,
    email VARCHAR(100) UNIQUE,
    age INT CHECK (age >= 18),
    status VARCHAR(20) DEFAULT 'Active'
);

DESCRIBE constraint_demo_overview;

**Expected result / explanation**

The table combines several constraints. `DESCRIBE` displays the table structure; constraint enforcement is tested by inserting data in the following sections.

## Topic 33: NOT NULL Constraint

`NOT NULL` ensures that a column must contain a value. It prevents a row from being inserted or updated with `NULL` in that column.

Use it for required information such as a person's name or an order date. An empty string (`''`) is not the same as `NULL`; `NOT NULL` does not reject empty strings by itself.

### SQL Example — Topic 33

In [ ]:
CREATE TABLE constraint_demo_not_null (
    student_id INT,
    student_name VARCHAR(80) NOT NULL,
    department VARCHAR(50) NOT NULL
);

INSERT INTO constraint_demo_not_null
    (student_id, student_name, department)
VALUES
    (1, 'Amit', 'Computer Science'),
    (2, 'Riya', 'Information Technology');

SELECT * FROM constraint_demo_not_null;

**Expected result / explanation**

Both rows provide values for the required columns. An insert that omits `student_name` or supplies `NULL` for it will fail.

## Topic 34: UNIQUE Constraint

`UNIQUE` ensures that values in a column, or a combination of columns, are not duplicated.

A table can have multiple `UNIQUE` constraints. In MySQL, a `UNIQUE` column can generally contain multiple `NULL` values because `NULL` is treated as an unknown value rather than an ordinary duplicate value.

### SQL Example — Topic 34

In [ ]:
CREATE TABLE constraint_demo_unique (
    user_id INT PRIMARY KEY,
    email VARCHAR(120) UNIQUE,
    username VARCHAR(50) UNIQUE
);

INSERT INTO constraint_demo_unique (user_id, email, username)
VALUES
    (1, 'amit@example.com', 'amit01'),
    (2, 'riya@example.com', 'riya02');

SELECT * FROM constraint_demo_unique;

**Expected result / explanation**

The email and username values must each be unique. Trying to insert an already-used email or username causes a duplicate-key error.

## Topic 35: PRIMARY KEY Constraint

A **primary key** uniquely identifies each row in a table.

Key properties:
- Values must be unique.
- A primary-key column cannot contain `NULL`.
- A table has one primary-key constraint, though it may contain more than one column (a composite primary key).
- A primary key can be referenced by a foreign key.

### SQL Example — Topic 35

In [ ]:
CREATE TABLE constraint_demo_primary_key (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department VARCHAR(50)
);

INSERT INTO constraint_demo_primary_key
    (employee_id, employee_name, department)
VALUES
    (101, 'Arjun', 'IT'),
    (102, 'Mina', 'HR');

SELECT * FROM constraint_demo_primary_key;

**Expected result / explanation**

`employee_id` is the primary key. Two rows cannot share the same employee ID, and the ID cannot be `NULL`.

## Topic 36: FOREIGN KEY Constraint

A **foreign key** creates and enforces a relationship between a child table and a referenced parent table. It prevents a child row from referring to a parent key that does not exist.

The referenced column must be indexed; it is commonly the parent table's primary key. The data types and signedness of corresponding columns should match.

`ON DELETE` and `ON UPDATE` actions can define what happens when a referenced parent key is changed or deleted.

### SQL Example — Topic 36

In [ ]:
CREATE TABLE constraint_demo_department (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(80) NOT NULL
);

CREATE TABLE constraint_demo_employee_fk (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department_id INT,
    CONSTRAINT fk_demo_employee_department
        FOREIGN KEY (department_id)
        REFERENCES constraint_demo_department(department_id)
);

INSERT INTO constraint_demo_department
    (department_id, department_name)
VALUES
    (10, 'IT'),
    (20, 'HR');

INSERT INTO constraint_demo_employee_fk
    (employee_id, employee_name, department_id)
VALUES
    (1, 'Kabir', 10),
    (2, 'Sana', 20);

SELECT
    e.employee_name,
    d.department_name
FROM constraint_demo_employee_fk AS e
JOIN constraint_demo_department AS d
    ON e.department_id = d.department_id;

**Expected result / explanation**

Insert parent departments before inserting employees that reference them. An employee cannot reference a department ID that is absent from the parent table. A `NULL` foreign-key value is permitted here because the child column was not declared `NOT NULL`.

## Topic 37: CHECK Constraint

`CHECK` validates a Boolean condition for each inserted or updated row. If the condition is false, MySQL rejects the row.

MySQL enforces `CHECK` constraints from version 8.0.16 onward. A check condition that evaluates to `UNKNOWN` because of `NULL` does not fail the check, so combine `CHECK` with `NOT NULL` when a value is mandatory.

### SQL Example — Topic 37

In [ ]:
CREATE TABLE constraint_demo_check (
    member_id INT PRIMARY KEY,
    member_name VARCHAR(80) NOT NULL,
    age INT NOT NULL,
    CONSTRAINT chk_demo_member_age CHECK (age >= 18)
);

INSERT INTO constraint_demo_check (member_id, member_name, age)
VALUES
    (1, 'Nila', 21),
    (2, 'Rohan', 30);

SELECT * FROM constraint_demo_check;

**Expected result / explanation**

The condition `age >= 18` permits the sample rows. An age below 18 is rejected by a MySQL version that enforces CHECK constraints.

## Topic 38: DEFAULT Constraint

`DEFAULT` supplies a value when an `INSERT` statement does not provide a value for that column.

You can also explicitly request the default using the `DEFAULT` keyword. A default is not the same as `NOT NULL`: unless separately constrained, a column may still accept `NULL` when `NULL` is explicitly supplied.

### SQL Example — Topic 38

In [ ]:
CREATE TABLE constraint_demo_default (
    task_id INT PRIMARY KEY,
    task_name VARCHAR(100) NOT NULL,
    task_status VARCHAR(20) DEFAULT 'Pending'
);

INSERT INTO constraint_demo_default (task_id, task_name)
VALUES (1, 'Prepare report');

INSERT INTO constraint_demo_default (task_id, task_name, task_status)
VALUES (2, 'Review report', DEFAULT);

SELECT * FROM constraint_demo_default;

**Expected result / explanation**

Both rows receive `Pending` as their task status because the insert omits the column or explicitly requests its default.

## Topic 39: AUTO_INCREMENT

`AUTO_INCREMENT` generates a new numeric value when a row is inserted without an explicit value for that column.

Common properties:
- The column must be indexed (typically it is the primary key).
- A table can have only one `AUTO_INCREMENT` column.
- It is commonly used for surrogate identifiers.
- The generated value can be retrieved in MySQL using `LAST_INSERT_ID()` on the same connection.

### SQL Example — Topic 39

In [ ]:
CREATE TABLE constraint_demo_auto_increment (
    id INT AUTO_INCREMENT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL
);

INSERT INTO constraint_demo_auto_increment (product_name)
VALUES ('Keyboard');

INSERT INTO constraint_demo_auto_increment (product_name)
VALUES ('Mouse');

SELECT * FROM constraint_demo_auto_increment;

SELECT LAST_INSERT_ID() AS last_generated_id;

**Expected result / explanation**

The database generates the IDs because the insert statements omit `id`. `LAST_INSERT_ID()` returns the most recently generated auto-increment value for the current connection.

## Topic 40: Composite Key

A **composite key** is a key made from two or more columns. A composite primary key uniquely identifies a row using the combined values.

It is useful when no single column is sufficient to identify a record, such as a student-course enrollment. Each column may repeat individually, but the combination must be unique.

### SQL Example — Topic 40

In [ ]:
CREATE TABLE constraint_demo_enrollment (
    student_id INT,
    course_id INT,
    enrollment_date DATE NOT NULL,
    PRIMARY KEY (student_id, course_id)
);

INSERT INTO constraint_demo_enrollment
    (student_id, course_id, enrollment_date)
VALUES
    (1, 101, '2026-01-10'),
    (1, 102, '2026-01-11'),
    (2, 101, '2026-01-12');

SELECT * FROM constraint_demo_enrollment;

**Expected result / explanation**

The same student can enroll in different courses, and the same course can contain different students. The pair `(student_id, course_id)` cannot be repeated.

## Topic 41: Candidate Key

A **candidate key** is a minimal set of column(s) that uniquely identifies each row.

- A table can have more than one candidate key.
- Every candidate key must be unique.
- It must be minimal: removing any column would make it no longer unique.
- One candidate key is selected as the primary key; the remaining candidate keys become alternate keys.

### SQL Example — Topic 41

In [ ]:
CREATE TABLE constraint_demo_candidate (
    person_id INT PRIMARY KEY,
    national_id VARCHAR(30) NOT NULL UNIQUE,
    email VARCHAR(120) NOT NULL UNIQUE,
    full_name VARCHAR(100) NOT NULL
);

INSERT INTO constraint_demo_candidate
    (person_id, national_id, email, full_name)
VALUES
    (1, 'NID-1001', 'a@example.com', 'Asha');

SELECT * FROM constraint_demo_candidate;

**Expected result / explanation**

In this design, `person_id`, `national_id`, and `email` are each minimal unique identifiers, so each is a candidate key. `person_id` is chosen as the primary key; the other two are alternate keys.

## Topic 42: Alternate Key

An **alternate key** is a candidate key that was not selected as the table's primary key.

For example, if `person_id`, `email`, and `phone_number` can each uniquely identify a person, the database designer chooses one as the primary key. The other candidate keys are alternate keys.

In SQL, alternate keys are commonly implemented using `UNIQUE` constraints.

### SQL Example — Topic 42

In [ ]:
CREATE TABLE constraint_demo_alternate (
    customer_id INT PRIMARY KEY,
    email VARCHAR(120) NOT NULL UNIQUE,
    phone_number VARCHAR(20) NOT NULL UNIQUE,
    customer_name VARCHAR(100) NOT NULL
);

INSERT INTO constraint_demo_alternate
    (customer_id, email, phone_number, customer_name)
VALUES
    (501, 'meera@example.com', '9000000001', 'Meera');

SELECT * FROM constraint_demo_alternate;

**Expected result / explanation**

`customer_id` is the chosen primary key. `email` and `phone_number` are alternate keys because they are candidate keys that were not chosen as the primary key.

## Topic 43: Super Key

A **super key** is any set of one or more columns that uniquely identifies a row. It may contain extra, unnecessary columns.

Every candidate key is a super key, but not every super key is a candidate key. A candidate key is a *minimal* super key.

For example, if `person_id` is unique, then `{person_id}` and `{person_id, full_name}` are both super keys. Only `{person_id}` is minimal.

### SQL Example — Topic 43

In [ ]:
CREATE TABLE constraint_demo_super_key (
    person_id INT PRIMARY KEY,
    full_name VARCHAR(100) NOT NULL,
    city VARCHAR(60)
);

INSERT INTO constraint_demo_super_key
    (person_id, full_name, city)
VALUES
    (1, 'Ishita Sen', 'Kolkata'),
    (2, 'Dev Roy', 'Howrah');

SELECT person_id, full_name, city
FROM constraint_demo_super_key;

**Expected result / explanation**

The primary key `person_id` guarantees uniqueness. Therefore `{person_id}` is a candidate key, while `{person_id, full_name}` is a super key with an unnecessary extra attribute. Super key is a relational-design concept; it does not require a separate SQL keyword.

## Topic 44: Natural Key

A **natural key** is a real-world attribute (or combination of attributes) that already has meaning outside the database and can identify a record.

Examples include an ISBN for a book or a government-issued identifier where its use is appropriate and lawful.

Advantages:
- Has real-world meaning.
- May avoid storing a separate identifier.

Disadvantages:
- Business rules or values can change.
- Values may be long or awkward to reference.
- Some natural identifiers are sensitive and should not be exposed unnecessarily.

### SQL Example — Topic 44

In [ ]:
CREATE TABLE constraint_demo_natural_key (
    isbn CHAR(13) PRIMARY KEY,
    book_title VARCHAR(200) NOT NULL,
    author_name VARCHAR(120)
);

INSERT INTO constraint_demo_natural_key
    (isbn, book_title, author_name)
VALUES
    ('9780132350884', 'Clean Code', 'Robert C. Martin');

SELECT * FROM constraint_demo_natural_key;

**Expected result / explanation**

Here, ISBN is used as a natural primary key because it is a meaningful identifier assigned to a book edition. In a production system, validate ISBN format and consider whether it is the best stable identifier for the application's needs.

## Topic 45: Surrogate Key

A **surrogate key** is an artificial identifier created for database use. It has no business meaning and is commonly generated using `AUTO_INCREMENT`.

Advantages:
- Usually short and stable.
- Does not depend on changing business attributes.
- Convenient for foreign-key relationships.

Disadvantages:
- Does not carry real-world meaning.
- A separate `UNIQUE` constraint may still be needed to enforce business uniqueness.

A common design uses a surrogate primary key plus a `UNIQUE` constraint on a natural business identifier.

### SQL Example — Topic 45

In [ ]:
CREATE TABLE constraint_demo_surrogate_key (
    book_id INT AUTO_INCREMENT PRIMARY KEY,
    isbn CHAR(13) NOT NULL UNIQUE,
    book_title VARCHAR(200) NOT NULL
);

INSERT INTO constraint_demo_surrogate_key (isbn, book_title)
VALUES
    ('9780132350884', 'Clean Code'),
    ('9780596007126', 'Head First Design Patterns');

SELECT * FROM constraint_demo_surrogate_key;

**Expected result / explanation**

`book_id` is the surrogate primary key. `isbn` remains unique to prevent the same book edition from being entered twice. This pattern combines a stable internal identifier with a meaningful business key.

## Quick Revision: Constraints

| Constraint | Main purpose |
|---|---|
| `NOT NULL` | Requires a value |
| `UNIQUE` | Prevents duplicate values |
| `PRIMARY KEY` | Uniquely identifies each row; no `NULL` |
| `FOREIGN KEY` | Enforces a valid parent-child reference |
| `CHECK` | Validates a condition |
| `DEFAULT` | Supplies a value when omitted |
| `AUTO_INCREMENT` | Generates numeric identifiers |

## Quick Revision: Keys

| Key | Meaning |
|---|---|
| Super key | Any column set that uniquely identifies a row |
| Candidate key | Minimal super key |
| Primary key | Candidate key selected as the main identifier |
| Alternate key | Candidate key not selected as primary |
| Composite key | Key containing multiple columns |
| Natural key | Meaningful real-world identifier |
| Surrogate key | Artificial database-generated identifier |

## Important Differences

### Candidate Key vs Super Key
- Every candidate key is a super key.
- A candidate key is minimal; a super key may contain extra columns.

### Primary Key vs UNIQUE
- A table has one primary-key constraint; it can have multiple `UNIQUE` constraints.
- A primary key cannot contain `NULL`.
- A `UNIQUE` column in MySQL can generally contain multiple `NULL` values.

### Natural Key vs Surrogate Key
- Natural key: meaningful real-world value, such as ISBN.
- Surrogate key: artificial identifier, such as an auto-generated integer.

## Practice Checklist

- [ ] Explain why constraints are used.
- [ ] Create a table using `NOT NULL` and `UNIQUE`.
- [ ] Define a primary key and a foreign key.
- [ ] Explain `CHECK`, `DEFAULT`, and `AUTO_INCREMENT`.
- [ ] Create a composite primary key.
- [ ] Differentiate super key and candidate key.
- [ ] Explain primary key and alternate key.
- [ ] Compare natural and surrogate keys.

**End of Part 03 — Topics 32–45.**